# 01 - Statistical Analysis with SciPy

Use SQL Server data to calculate descriptive statistics, correlations, and hypothesis tests for financial supervision indicators.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

from analysis_utils import ensure_output_dir, load_indicator_data

ENV_FILE = ".env"  # Windows learners can change this to ".env.windows"
output_dir = ensure_output_dir()
df = load_indicator_data(ENV_FILE)
df.head()

## Descriptive Statistics

In [ ]:
analysis_columns = [
    "LiquidityRatio",
    "CapitalAdequacyRatio",
    "NplRatio",
    "TransactionValueLSL",
    "CreditGrowthRate",
    "InflationRate",
    "InterbankRate",
]

summary = df[analysis_columns].describe().T
summary["variance"] = df[analysis_columns].var(numeric_only=True)
summary.to_csv(output_dir / "statistical_summary.csv")
summary

## Correlation Analysis

In [ ]:
correlations = df[analysis_columns].corr()
liquidity_npl_corr, liquidity_npl_pvalue = stats.pearsonr(df["LiquidityRatio"], df["NplRatio"])

print(f"Liquidity vs NPL correlation: {liquidity_npl_corr:.3f}")
print(f"p-value: {liquidity_npl_pvalue:.6f}")
correlations

## Hypothesis Test

Question: is average credit growth different between stressed and non-stressed observations?

In [ ]:
stressed = df.loc[df["StressFlag"] == 1, "CreditGrowthRate"]
not_stressed = df.loc[df["StressFlag"] == 0, "CreditGrowthRate"]

test_result = stats.ttest_ind(stressed, not_stressed, equal_var=False)

print(f"Stressed mean credit growth: {stressed.mean():.4f}")
print(f"Non-stressed mean credit growth: {not_stressed.mean():.4f}")
print(f"t-statistic: {test_result.statistic:.3f}")
print(f"p-value: {test_result.pvalue:.6f}")

## Group Comparison

In [ ]:
group_summary = (
    df.groupby("InstitutionType")
    .agg(
        rows=("ObservationID", "count"),
        mean_liquidity=("LiquidityRatio", "mean"),
        mean_npl=("NplRatio", "mean"),
        stress_rate=("StressFlag", "mean"),
        avg_transaction_value=("TransactionValueLSL", "mean"),
    )
    .sort_values("stress_rate", ascending=False)
)

group_summary.to_csv(output_dir / "institution_type_summary.csv")
group_summary